# Gobernanza, Seguridad y Cumplimiento (Senior)

Objetivos:
- Definir un marco práctico de gobernanza alineado a negocio.
- Implementar controles de seguridad (RBAC, ABAC, clasificación, cifrado).
- Integrar cumplimiento (GDPR, CCPA) en el ciclo de datos.
- Diseñar auditoría y trazabilidad (lineage + logging).

## 1. Gobernanza: Componentes Clave

Pilares:
1. Políticas (definiciones y alcance).
2. Roles y Responsabilidades (stewards, custodians, owners).
3. Procesos (ingestión, calidad, acceso, cambio de esquema).
4. Métricas (calidad, frescura, cumplimiento de SLAs).
5. Herramientas (catálogo, lineage, quality suite).

Modelo RACI breve ejemplo: 
- Steward: Responsable de calidad y significado.
- Owner: Aprueba cambios y acceso.
- Custodian (TI): Implementa controles técnicos.

## 1.1. Roles y Responsabilidades Detalladas

**Roles de Gobernanza** (basado en DAMA-DMBOK):

| Rol | Responsabilidades | Ejemplo |
|-----|-------------------|---------|
| **Data Owner** | Define reglas de negocio, aprueba accesos, responsable del valor del dato | Gerente Comercial (owner de `clientes`) |
| **Data Steward** | Calidad, significado, metadatos; puente entre negocio y TI | Analista de Calidad |
| **Data Custodian** | Implementación técnica, backups, seguridad física/lógica | Equipo Infraestructura |
| **Data Governance Lead** | Estrategia, políticas, comité de gobierno, métricas | Arquitecto de Datos Senior |
| **Consumers** | Usuarios finales, analistas, científicos de datos | Equipo BI, Marketing |

**Matriz RACI (ejemplo para cambio de esquema):**

| Actividad | Owner | Steward | Custodian | Governance Lead | Consumers |
|-----------|-------|---------|-----------|-----------------|-----------|
| Proponer cambio | I | R | C | C | I |
| Evaluar impacto | C | A | R | C | I |
| Aprobar cambio | A | C | I | C | I |
| Implementar | I | I | R | I | I |
| Comunicar | I | R | C | A | I |

R=Responsable, A=Aprueba, C=Consultado, I=Informado


## 1.2. Políticas de Datos

**Política = regla/directriz que establece cómo se gestionan los datos.**

### Ejemplos de Políticas

1. **Política de Clasificación:**
   - Todo dataset debe clasificarse: Público, Interno, Confidencial, Regulado.
   - Clasificación por defecto = Interno hasta revisión.

2. **Política de Acceso:**
   - Acceso basado en "need-to-know" (mínimo privilegio).
   - Revisión trimestral de permisos.
   - Aprobación del Data Owner + Seguridad para datos Regulados.

3. **Política de Retención:**
   - Datos transaccionales: 7 años (compliance fiscal).
   - Logs: 90 días en caliente, 1 año en archivo.
   - PII: borrado dentro de 30 días de solicitud (GDPR).

4. **Política de Calidad:**
   - Silver y Gold requieren validación (quality gate) antes de publicación.
   - SLO de frescura: Gold diario listo antes de 07:00.
   - Alertas automáticas si calidad < umbral (ej. nulls > 5%).

### Enforcement

- Automatización: quality gates en CI/CD, RBAC en catálogo/warehouse.
- Monitoreo: dashboards de cumplimiento (% datasets clasificados, auditorías pendientes).
- Excepciones: proceso formal de aprobación y logging.


## 1.3. Procedimientos (Workflows)

**Procedimiento = secuencia de pasos para aplicar una política.**

### Ejemplo 1: Solicitud de Acceso a Dataset

1. Usuario completa formulario (dataset, justificación, alcance).
2. Sistema valida clasificación → si Regulado, requiere aprobación adicional.
3. Data Owner revisa y aprueba/rechaza.
4. Si aprobado, Custodian provisiona acceso (RBAC).
5. Auditoría registra acceso otorgado (quién, cuándo, por qué).
6. Revisión periódica (90 días) para renovar o revocar.

### Ejemplo 2: Cambio de Esquema (Breaking Change)

1. Propuesta con justificación e impacto estimado.
2. Steward evalúa impacto en downstream (lineage).
3. Notificación a consumidores afectados (mín. 2 semanas).
4. Governance Lead aprueba (committee si es crítico).
5. Implementación en dev → QA → staging → prod.
6. Post-mortem: documentar incidentes y ajustar proceso.

### Ejemplo 3: Proceso de Calidad (Quality Gate)

1. Dataset llega a zona Silver.
2. Ejecución automática de suite de calidad (nulls, duplicados, esquema).
3. Si falla test crítico → alerta + bloqueo de publicación a Gold.
4. Steward investiga y corrige origen (Bronze o ingesta).
5. Re-ejecución tras corrección.
6. Aprobación manual (si aplica) y promoción a Gold.


## 1.4. Herramientas de Gobernanza

| Categoría | Herramientas | Uso |
|-----------|--------------|-----|
| **Catálogo** | Atlan, Collibra, Alation, DataHub, Apache Atlas | Inventario, búsqueda, documentación, lineage |
| **Calidad** | Great Expectations, Soda, dbt tests, Monte Carlo | Validación automática, alertas, monitoreo drift |
| **Lineage** | OpenLineage, Marquez, dbt lineage, Snowflake lineage | Rastreo origen-destino, análisis de impacto |
| **Acceso/Seguridad** | Ranger, Privacera, Immuta, BigQuery IAM | RBAC/ABAC, masking dinámico, audit logs |
| **Workflow** | Airflow, Prefect, Dagster | Orquestación pipelines + hooks de calidad |

**Integración típica:**
- Catálogo ← metadatos técnicos (auto-discovery) + metadatos de negocio (manual).
- Quality suite → genera métricas → catálogo las expone.
- Lineage recolectado vía OpenLineage → Marquez → catálogo visualiza.


In [ ]:
# Simulación de métricas de gobierno
import pandas as pd

# Inventario de datasets (simulado)
datasets = pd.DataFrame({
    'dataset': ['bronze.ventas_raw', 'silver.ventas_limpias', 'gold.kpi_ventas', 'bronze.clientes_raw', 'silver.clientes'],
    'clasificacion': ['Interno', 'Interno', 'Confidencial', 'Regulado', 'Regulado'],
    'owner': ['Comercial', 'Comercial', 'Comercial', 'Comercial', 'Comercial'],
    'steward_asignado': [True, True, True, False, True],
    'calidad_validada': [False, True, True, False, True],
    'acceso_revisado_90d': [False, True, True, False, False]
})

# Métricas de cumplimiento
pct_steward = datasets['steward_asignado'].mean() * 100
pct_calidad = datasets['calidad_validada'].mean() * 100
pct_acceso_rev = datasets['acceso_revisado_90d'].mean() * 100

print('=== Dashboard de Gobierno ===')
print(f'Datasets con Steward asignado: {pct_steward:.0f}%')
print(f'Datasets con calidad validada: {pct_calidad:.0f}%')
print(f'Datasets con acceso revisado (90d): {pct_acceso_rev:.0f}%')
print('\nDatasets sin Steward:')
print(datasets[~datasets['steward_asignado']]['dataset'].tolist())


## 2. Clasificación y Protección de Datos

Clasificaciones típicas: Público, Interno, Confidencial, Regulados (PII/PCI/PHI).

Controles:
- Data Masking (enmascaramiento dinámico).
- Tokenización (reemplazo reversible controlado).
- Pseudonimización (irrevocable o difícil de revertir).
- Cifrado at-rest (KMS) y in-transit (TLS).

Ejemplo de mapeo PII: nombre, email, teléfono, dirección.

In [ ]:
# Ejemplo simple de enmascaramiento de PII en pandas
import pandas as pd
df = pd.DataFrame({
  'customer_id':[1,2,3],
  'nombre':['Ana García','Luis Pérez','María López'],
  'email':['ana@example.com','luis@example.com','maria@example.com'],
  'telefono':['555-1111','555-2222','555-3333'],
  'importe_total':[1200.50,300.00,450.75]
})
def mask_email(e):
    user, domain = e.split('@'); return user[0]+'***@'+domain
def mask_nombre(n):
    partes = n.split(' '); return partes[0][0]+'*** '+partes[-1][0]+'***'
masked = df.copy()
masked['email'] = masked['email'].apply(mask_email)
masked['nombre'] = masked['nombre'].apply(mask_nombre)
masked['telefono'] = '***-****'
print('Original:
', df)
print('
Enmascarado:
', masked)

## 3. Modelos de Acceso: RBAC vs ABAC

RBAC (Role-Based Access Control): Permisos por rol.
ABAC (Attribute-Based Access Control): Evaluación de atributos (usuario, recurso, contexto).

Ejemplo ABAC: Permitir acceso si (pais=="CO" AND clearance=="ALTO").

Recomendación: Híbrido → RBAC para simplicidad + ABAC para excepciones/reglas finas.

## 4. Cumplimiento (GDPR / CCPA)

Elementos clave:
- Minimización de datos.
- Derecho al olvido / borrado selectivo.
- Consentimiento explícito y revocación.
- Registro de procesamientos (auditoría).

Flujo derecho al olvido: solicitud → verificación → localización de registros → borrado → evidencia registro.

In [ ]:
# Simulación 'derecho al olvido' simple
solicitudes_borrado = [2]  # customer_id a eliminar
post_borrado = df[~df['customer_id'].isin(solicitudes_borrado)]
print('Antes:
', df)
print('
Después de borrado (right to be forgotten):
', post_borrado)

## 5. Auditoría y Lineage

Auditoría: Registro de quién accede, qué modifica, cuándo.
Lineage: Visualización de origen→transformaciones→destino.

Prácticas:
- IDs de ejecución (run_id).
- Trazabilidad de columnas (column-level lineage).
- Registro de versiones de esquema.

Herramientas: OpenLineage, Marquez, Atlas.

## 6. Métricas de Gobernanza

Ejemplos:
- % datasets con steward asignado.
- % cobertura de catálogo.
- Tiempo promedio de aprobación de acceso.
- % cumplimiento de políticas (sin PII sin cifrar).

## 7. Ejercicio

Define para tu organización:
1. Tabla RACI para 3 dominios de datos.
2. Política de clasificación (4 niveles) y controles asociados.
3. Flujo operativo para derecho al olvido (actores y pasos).
4. Métricas iniciales de gobernanza (mínimo 4).

In [ ]:
plantilla = '''
# Solución (borrador)
RACI: [dominio -> roles]
Clasificación: [niveles + controles]
Borrado: [flujo]
Métricas: [lista]
'''
print(plantilla)

## 8. Resumen

- Gobernanza = personas + políticas + procesos + herramientas.
- Seguridad: combinar RBAC y ABAC + protección PII.
- Cumplimiento requiere operacionalizar derechos y evidencias.
- Auditoría y lineage habilitan confianza y trazabilidad.

---
Referencias:
- DAMA-DMBOK 2.0
- ISO/IEC 27001
- GDPR / CCPA textos oficiales
- NIST Privacy Framework

## 8. Ejercicio Integrador: Gobierno en Retail Omnicanal

**Contexto:** RetailCorp consolida datos de ERP, e-commerce, CRM y POS en un Lakehouse (Databricks). Problemas identificados: datasets sin clasificar, accesos descontrolados, calidad inconsistente, sin lineage.

**Dominio:** Ventas

**Datasets:**
1. `bronze.pos_transacciones_raw` (5M filas, con email_cliente opcional)
2. `bronze.ecommerce_pedidos_raw` (2M filas, con nombre/email/dirección)
3. `silver.ventas_unificadas` (6.8M filas, cliente_id hasheado)
4. `gold.kpi_ventas_diarias` (365 filas, sin PII)
5. `gold.clientes_perfil_360` (1.2M filas, con email/teléfono/historial)

**Ver caso completo:** `../04-recursos/casos-uso/caso_integrador_gobierno_retail.md`


### Tareas del Ejercicio

**2.1. Asignar Roles:**
- Para cada dataset, asigna Data Owner, Steward, Custodian.
- Considera: Gerente Comercial, Analista Calidad, Equipo Infra, Arquitecto Datos, Equipo BI, Marketing.

**2.2. Clasificar Datasets:**
- Usa categorías: Público, Interno, Confidencial, Regulado.
- Justifica cada clasificación (presencia de PII, sensibilidad).

**2.3. Matriz RACI (cambio de esquema):**
- Escenario: agregar columna `metodo_pago` a `bronze.ecommerce_pedidos_raw`.
- Actividades: Proponer, Evaluar impacto, Notificar, Aprobar, Implementar, Documentar.
- Asigna R/A/C/I a cada rol involucrado.

**2.4. Construir Lineage:**
- Dibuja grafo: fuentes → Bronze → Silver → Gold → consumidores.
- Indica transformaciones en cada paso.

**2.5. Política de Calidad:**
- Define 3 reglas para `silver.ventas_unificadas` (ej. no duplicados por pedido_id).
- SLO: nulls_cliente_id < 1%.
- Procedimiento: ¿qué pasa si falla quality gate?

**2.6. Workflow de Acceso:**
- Marketing solicita acceso a `gold.clientes_perfil_360` (Regulado).
- Diseña flujo de 6-7 pasos (solicitud → aprobaciones → provisión → auditoría → revisión).


In [ ]:
# Plantilla para solución (ejemplo parcial)
import pandas as pd

# 2.1 Roles (tabla ejemplo)
roles = pd.DataFrame({
    'dataset': ['bronze.pos_transacciones_raw', 'bronze.ecommerce_pedidos_raw', 
                'silver.ventas_unificadas', 'gold.kpi_ventas_diarias', 'gold.clientes_perfil_360'],
    'owner': ['Gerente Comercial', 'Gerente Comercial', 'Gerente Comercial', 'Equipo BI', 'Equipo Marketing'],
    'steward': ['Analista Calidad', 'Analista Calidad', 'Analista Calidad', 'Analista Calidad', 'Analista Calidad'],
    'custodian': ['Equipo Infra']*5
})

# 2.2 Clasificación
clasificacion = pd.DataFrame({
    'dataset': roles['dataset'],
    'clasificacion': ['Regulado', 'Regulado', 'Confidencial', 'Interno', 'Regulado'],
    'justificacion': [
        'Contiene email_cliente (PII)',
        'Contiene nombre, email, dirección (PII)',
        'cliente_id hasheado (pseudo-PII), datos estratégicos',
        'Agregados sin PII, uso interno',
        'Email, teléfono, historial (PII completo)'
    ]
})

print('=== Roles ===')
print(roles.to_string(index=False))
print('\n=== Clasificación ===')
print(clasificacion.to_string(index=False))

# Completa el resto de tareas (RACI, lineage, reglas de calidad, workflow)
